# Pon.Bike Demo - Data Ingestion

This notebook generates the POC database for Pon.Bike competitive intelligence demo in the Dutch bicycle market.

In [ ]:
import os
from snowflake.snowpark import Session

session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "oregon_tp")).create()
print(f"Connected as: {session.get_current_user()}")
print(f"Role: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")

## Step 1: Create Database and Schema

In [ ]:
session.sql("CREATE OR REPLACE DATABASE PON_BIKE_DEMO").collect()
session.sql("CREATE OR REPLACE SCHEMA PON_BIKE_DEMO.ANALYTICS").collect()
session.sql("USE SCHEMA PON_BIKE_DEMO.ANALYTICS").collect()
print("Database and schema created.")

## Step 2: Create Tables

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE DIM_BRANDS (
    BRAND_ID INT PRIMARY KEY,
    BRAND_NAME VARCHAR(200) NOT NULL,
    PARENT_COMPANY VARCHAR(100),
    HEADQUARTERS VARCHAR(100),
    FOUNDED_YEAR INT,
    BRAND_TYPE VARCHAR(50),
    PRICE_TIER VARCHAR(20),
    IS_PON_BRAND BOOLEAN DEFAULT FALSE,
    PRIMARY_CATEGORY VARCHAR(50),
    NL_DEALER_COUNT INT
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE DIM_BIKE_CATEGORIES (
    CATEGORY_ID INT PRIMARY KEY,
    CATEGORY_NAME VARCHAR(100) NOT NULL,
    SEGMENT VARCHAR(50),
    AVG_PRICE_EUR INT,
    GROWTH_TREND VARCHAR(20)
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE BRIDGE_BRAND_CATEGORIES (
    BRAND_ID INT,
    CATEGORY_ID INT,
    PRIMARY KEY (BRAND_ID, CATEGORY_ID)
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE FACT_BRAND_METRICS (
    BRAND_ID INT,
    YEAR_MONTH DATE,
    NL_UNITS_SOLD INT,
    NL_REVENUE_EUR INT,
    AVG_SELLING_PRICE_EUR INT,
    NL_MARKET_SHARE_PCT FLOAT,
    ONLINE_SENTIMENT_SCORE FLOAT,
    NEW_MODEL_LAUNCHES INT,
    PRIMARY KEY (BRAND_ID, YEAR_MONTH)
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE REVIEWS (
    REVIEW_ID INT PRIMARY KEY,
    BRAND_ID INT NOT NULL,
    BIKE_CATEGORY VARCHAR(100),
    REVIEWER_NAME VARCHAR(100),
    REVIEWER_CITY VARCHAR(100),
    RATING INT,
    REVIEW_DATE DATE,
    REVIEW_TITLE VARCHAR(500),
    REVIEW_TEXT VARCHAR(4000),
    PURCHASE_TYPE VARCHAR(50),
    USAGE_TYPE VARCHAR(50)
)
""").collect()

print("All tables created successfully.")

## Step 3: Generate Brand Data

In [ ]:
import pandas as pd

brands_data = [
    (1, "Gazelle", "Pon.Bike", "Netherlands", 1892, "City/Commuter", "Mid-Range", True, "E-bike City", 350),
    (2, "Cannondale", "Pon.Bike", "USA", 1971, "Performance", "Premium", True, "Road", 120),
    (3, "Santa Cruz", "Pon.Bike", "USA", 1993, "Performance", "Luxury", True, "Mountain Bike", 25),
    (4, "Cervelo", "Pon.Bike", "Canada", 1995, "Performance", "Luxury", True, "Road", 30),
    (5, "Kalkhoff", "Pon.Bike", "Germany", 1919, "City/Commuter", "Mid-Range", True, "E-bike City", 180),
    (6, "Focus", "Pon.Bike", "Germany", 1992, "Performance", "Premium", True, "Road", 85),
    (7, "Urban Arrow", "Pon.Bike", "Netherlands", 2010, "Cargo", "Premium", True, "Cargo", 60),
    (8, "Veloretti", "Pon.Bike", "Netherlands", 2013, "Lifestyle", "Mid-Range", True, "City/Commuter", 15),
    (9, "Schwinn", "Pon.Bike", "USA", 1895, "Lifestyle", "Budget", True, "City/Commuter", 40),

    (10, "Trek", "Trek Inc.", "USA", 1976, "Full-Range", "Premium", False, "Road", 200),
    (11, "Specialized", "Independent", "USA", 1974, "Performance", "Premium", False, "Road", 150),
    (12, "Giant", "Giant Group", "Taiwan", 1972, "Full-Range", "Mid-Range", False, "Road", 180),
    (13, "Cube", "Pending System", "Germany", 1993, "Full-Range", "Mid-Range", False, "Mountain Bike", 160),
    (14, "Batavus", "Accell Group", "Netherlands", 1904, "City/Commuter", "Mid-Range", False, "E-bike City", 300),
    (15, "Sparta", "Accell Group", "Netherlands", 1917, "City/Commuter", "Mid-Range", False, "E-bike City", 280),
    (16, "Cortina", "Kruitbosch", "Netherlands", 2007, "Lifestyle", "Budget", False, "City/Commuter", 250),
    (17, "Riese & Muller", "Independent", "Germany", 1993, "Cargo", "Luxury", False, "Cargo", 45),
    (18, "Babboe", "Accell Group", "Netherlands", 2007, "Cargo", "Mid-Range", False, "Cargo", 120),
    (19, "Haibike", "Accell Group", "Germany", 1995, "Performance", "Premium", False, "E-MTB", 90),
    (20, "Scott", "Scott Sports", "Switzerland", 1958, "Performance", "Premium", False, "Mountain Bike", 100),
    (21, "Merida", "Merida Industry", "Taiwan", 1972, "Full-Range", "Mid-Range", False, "Road", 110),
    (22, "BMC", "Independent", "Switzerland", 1994, "Performance", "Luxury", False, "Road", 35),
    (23, "Canyon", "Independent", "Germany", 2002, "Performance", "Premium", False, "Road", 0),
    (24, "VanMoof", "Lavoie", "Netherlands", 2009, "Lifestyle", "Premium", False, "E-bike City", 5),
    (25, "Koga", "Accell Group", "Netherlands", 1974, "City/Commuter", "Premium", False, "Touring", 140),
]

pdf = pd.DataFrame(brands_data, columns=[
    "BRAND_ID", "BRAND_NAME", "PARENT_COMPANY", "HEADQUARTERS", "FOUNDED_YEAR",
    "BRAND_TYPE", "PRICE_TIER", "IS_PON_BRAND", "PRIMARY_CATEGORY", "NL_DEALER_COUNT"
])

df = session.create_dataframe(pdf)
df.write.mode("overwrite").save_as_table("DIM_BRANDS")
print(f"Inserted {session.table('DIM_BRANDS').count()} brands")

## Step 4: Generate Bike Categories

In [ ]:
categories_data = [
    (1, "City/Commuter", "Urban Mobility", 800, "Stable"),
    (2, "E-bike City", "Urban Mobility", 2500, "Growing"),
    (3, "E-bike Trekking", "Urban Mobility", 3200, "Growing"),
    (4, "Road", "Performance", 3000, "Stable"),
    (5, "Mountain Bike", "Performance", 2500, "Stable"),
    (6, "E-MTB", "Performance", 4500, "Growing"),
    (7, "Gravel", "Performance", 2800, "Growing"),
    (8, "Cargo", "Utility", 3500, "Growing"),
    (9, "Kids", "Recreation", 400, "Stable"),
    (10, "Touring", "Urban Mobility", 1800, "Declining"),
    (11, "Speed Pedelec", "Urban Mobility", 4000, "Growing"),
    (12, "Folding", "Urban Mobility", 1200, "Stable"),
]

pdf_categories = pd.DataFrame(categories_data, columns=[
    "CATEGORY_ID", "CATEGORY_NAME", "SEGMENT", "AVG_PRICE_EUR", "GROWTH_TREND"
])
session.create_dataframe(pdf_categories).write.mode("overwrite").save_as_table("DIM_BIKE_CATEGORIES")
print(f"Inserted {session.table('DIM_BIKE_CATEGORIES').count()} bike categories")

## Step 5: Create Brand-Category Mappings

In [ ]:
brand_categories = [
    (1, 1), (1, 2), (1, 3), (1, 9),
    (2, 4), (2, 5), (2, 7), (2, 6),
    (3, 5), (3, 7),
    (4, 4), (4, 7),
    (5, 1), (5, 2), (5, 3),
    (6, 4), (6, 5), (6, 7), (6, 6),
    (7, 8),
    (8, 1), (8, 2),
    (9, 1), (9, 9),
    (10, 1), (10, 2), (10, 4), (10, 5), (10, 6), (10, 7), (10, 8), (10, 9),
    (11, 4), (11, 5), (11, 6), (11, 7), (11, 2),
    (12, 1), (12, 2), (12, 4), (12, 5), (12, 6), (12, 7), (12, 9),
    (13, 4), (13, 5), (13, 6), (13, 2), (13, 7), (13, 9),
    (14, 1), (14, 2), (14, 3), (14, 9),
    (15, 1), (15, 2), (15, 3),
    (16, 1), (16, 9),
    (17, 8), (17, 2), (17, 11),
    (18, 8),
    (19, 6), (19, 5), (19, 2),
    (20, 4), (20, 5), (20, 6), (20, 7),
    (21, 4), (21, 5), (21, 6), (21, 2),
    (22, 4), (22, 7),
    (23, 4), (23, 5), (23, 7), (23, 6),
    (24, 2),
    (25, 1), (25, 10), (25, 2),
]

pdf_bc = pd.DataFrame(brand_categories, columns=["BRAND_ID", "CATEGORY_ID"])
session.create_dataframe(pdf_bc).write.mode("overwrite").save_as_table("BRIDGE_BRAND_CATEGORIES")
print(f"Created {len(brand_categories)} brand-category mappings")

## Step 6: Generate Monthly Brand Metrics

In [ ]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import random

random.seed(42)

brands_df = session.table("DIM_BRANDS").to_pandas()

base_monthly_units = {
    "Gazelle": 8000, "Cannondale": 1200, "Santa Cruz": 150, "Cervelo": 200,
    "Kalkhoff": 3000, "Focus": 600, "Urban Arrow": 400, "Veloretti": 800, "Schwinn": 500,
    "Trek": 5000, "Specialized": 3500, "Giant": 6000, "Cube": 4000,
    "Batavus": 7000, "Sparta": 5500, "Cortina": 4500, "Riese & Muller": 300,
    "Babboe": 600, "Haibike": 800, "Scott": 1500, "Merida": 2000,
    "BMC": 200, "Canyon": 2500, "VanMoof": 100, "Koga": 1800,
}

avg_price = {"Budget": 600, "Mid-Range": 1800, "Premium": 3500, "Luxury": 6000}

start_month = datetime(2023, 1, 1)
end_month = datetime(2026, 2, 1)

metrics_data = []
current_month = start_month

while current_month <= end_month:
    month_total_units = 0
    month_brand_units = {}

    for _, brand in brands_df.iterrows():
        brand_id = int(brand["BRAND_ID"])
        brand_name = brand["BRAND_NAME"]
        price_tier = brand["PRICE_TIER"]

        base = base_monthly_units.get(brand_name, 500)

        month_num = current_month.month
        if month_num in [4, 5, 6]:
            seasonality = 1.4
        elif month_num in [7, 8]:
            seasonality = 1.2
        elif month_num in [11, 12, 1, 2]:
            seasonality = 0.6
        else:
            seasonality = 1.0

        noise = random.uniform(0.85, 1.15)
        units = int(base * seasonality * noise)
        month_brand_units[brand_id] = units
        month_total_units += units

    for brand_id, units in month_brand_units.items():
        brand_row = brands_df[brands_df["BRAND_ID"] == brand_id].iloc[0]
        price_tier = brand_row["PRICE_TIER"]

        asp = int(avg_price.get(price_tier, 1800) * random.uniform(0.9, 1.1))
        revenue = units * asp
        market_share = round(units / month_total_units * 100, 2)
        sentiment = round(random.uniform(3.5, 4.8), 2)
        new_models = random.choices([0, 0, 0, 0, 0, 1, 1, 2], k=1)[0]

        metrics_data.append((
            brand_id,
            current_month.strftime("%Y-%m-01"),
            units,
            revenue,
            asp,
            market_share,
            sentiment,
            new_models
        ))

    current_month += relativedelta(months=1)

pdf_metrics = pd.DataFrame(metrics_data, columns=[
    "BRAND_ID", "YEAR_MONTH", "NL_UNITS_SOLD", "NL_REVENUE_EUR",
    "AVG_SELLING_PRICE_EUR", "NL_MARKET_SHARE_PCT", "ONLINE_SENTIMENT_SCORE", "NEW_MODEL_LAUNCHES"
])

session.create_dataframe(pdf_metrics).write.mode("overwrite").save_as_table("FACT_BRAND_METRICS")
print(f"Generated {session.table('FACT_BRAND_METRICS').count()} monthly metrics records")

## Step 7: Generate 250 Product Reviews

In [ ]:
from datetime import datetime, timedelta
import random

random.seed(42)

brands_df = session.table("DIM_BRANDS").to_pandas()
categories_df = session.table("DIM_BIKE_CATEGORIES").to_pandas()
bridge_df = session.table("BRIDGE_BRAND_CATEGORIES").to_pandas()

reviewer_names = [
    "FietsFreak", "CyclingDutch", "BikeCommuter", "WielrennerNL", "E-BikeRijder",
    "JanW", "PieterK", "MariekV", "ThomasB", "SophieD", "BramM", "LisaH", "DaanR",
    "EmmaJ", "LucasP", "FleurS", "NielsT", "AnneG", "TimF", "SanneL",
    "VeloLover", "MTBNederland", "CargoParent", "GravelRider", "StadsFietser",
]

reviewer_cities = [
    "Amsterdam", "Rotterdam", "Utrecht", "Den Haag", "Eindhoven",
    "Groningen", "Tilburg", "Almere", "Breda", "Nijmegen",
    "Haarlem", "Arnhem", "Leiden", "Delft", "Maastricht",
]

purchase_types = ["New", "Used", "Lease/Subscription"]
purchase_weights = [65, 20, 15]

usage_types = ["Daily Commute", "Recreation", "Sport", "Family Transport"]
usage_weights = [40, 25, 20, 15]

start_date = datetime(2023, 1, 1)
end_date = datetime(2026, 2, 28)
date_range = (end_date - start_date).days

base_reviews_per_brand = {}
for _, brand in brands_df.iterrows():
    brand_id = int(brand["BRAND_ID"])
    if brand["BRAND_NAME"] in ["Trek", "Specialized", "Giant"]:
        base_reviews_per_brand[brand_id] = 18
    elif brand["BRAND_NAME"] in ["Gazelle", "Batavus", "Cube"]:
        base_reviews_per_brand[brand_id] = 15
    elif brand["IS_PON_BRAND"]:
        base_reviews_per_brand[brand_id] = 10
    else:
        base_reviews_per_brand[brand_id] = 8

total_reviews = sum(base_reviews_per_brand.values())
scale_factor = 275 / total_reviews
for brand_id in base_reviews_per_brand:
    base_reviews_per_brand[brand_id] = max(4, int(base_reviews_per_brand[brand_id] * scale_factor))

def get_rating_for_brand(brand_name, is_pon, brand_type):
    if brand_name in ["Trek", "Specialized"]:
        weights = [2, 5, 12, 38, 43]
    elif brand_name in ["Giant", "Cube", "Scott", "BMC"]:
        weights = [3, 6, 15, 36, 40]
    elif brand_name in ["Gazelle", "Kalkhoff"]:
        weights = [4, 8, 20, 35, 33]
    elif is_pon:
        weights = [5, 10, 22, 35, 28]
    else:
        weights = [4, 8, 18, 35, 35]
    return random.choices([1, 2, 3, 4, 5], weights=weights)[0]

cat_lookup = dict(zip(categories_df["CATEGORY_ID"], categories_df["CATEGORY_NAME"]))

reviews_metadata = []
for _, brand in brands_df.iterrows():
    brand_id = int(brand["BRAND_ID"])
    brand_name = brand["BRAND_NAME"]
    brand_type = brand["BRAND_TYPE"]
    price_tier = brand["PRICE_TIER"]
    is_pon = brand["IS_PON_BRAND"]

    brand_cats = bridge_df[bridge_df["BRAND_ID"] == brand_id]["CATEGORY_ID"].tolist()
    num_reviews = base_reviews_per_brand.get(brand_id, 6)

    for _ in range(num_reviews):
        rating = get_rating_for_brand(brand_name, is_pon, brand_type)
        cat_id = random.choice(brand_cats) if brand_cats else 1
        bike_category = cat_lookup.get(cat_id, brand["PRIMARY_CATEGORY"])
        reviewer_name = random.choice(reviewer_names) + str(random.randint(1, 999))
        reviewer_city = random.choice(reviewer_cities)
        review_date = (start_date + timedelta(days=random.randint(0, date_range))).strftime("%Y-%m-%d")
        purchase_type = random.choices(purchase_types, weights=purchase_weights)[0]
        usage_type = random.choices(usage_types, weights=usage_weights)[0]

        reviews_metadata.append((
            brand_id, brand_name, brand_type, price_tier, is_pon,
            bike_category, reviewer_name, reviewer_city, rating,
            review_date, purchase_type, usage_type
        ))

random.shuffle(reviews_metadata)
reviews_metadata = reviews_metadata[:250]

pdf_meta = pd.DataFrame(reviews_metadata, columns=[
    "BRAND_ID", "BRAND_NAME", "BRAND_TYPE", "PRICE_TIER", "IS_PON_BRAND",
    "BIKE_CATEGORY", "REVIEWER_NAME", "REVIEWER_CITY", "RATING",
    "REVIEW_DATE", "PURCHASE_TYPE", "USAGE_TYPE"
])
session.create_dataframe(pdf_meta).write.mode("overwrite").save_as_table("REVIEWS_STAGING")
print(f"Created staging table with {len(reviews_metadata)} review metadata records")

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE REVIEWS AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY REVIEW_DATE) AS REVIEW_ID,
    BRAND_ID,
    BIKE_CATEGORY,
    REVIEWER_NAME,
    REVIEWER_CITY,
    RATING,
    REVIEW_DATE::DATE AS REVIEW_DATE,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-5',
        'Generate a short product review title (5-10 words) for a ' || RATING || '-star review of a '
        || BRAND_NAME || ' ' || BIKE_CATEGORY || ' bicycle. '
        || 'Price tier: ' || PRICE_TIER || '. Usage: ' || USAGE_TYPE || '. '
        || CASE 
            WHEN RATING >= 4 THEN 'The reviewer loved it.'
            WHEN RATING = 3 THEN 'The reviewer had mixed feelings.'
            ELSE 'The reviewer was disappointed.'
           END
        || ' Return ONLY the title, no quotes or explanation.'
    ) AS REVIEW_TITLE,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-5',
        'Write a realistic product review (80-120 words) for a ' || RATING || '-star '
        || BRAND_NAME || ' ' || BIKE_CATEGORY || ' bicycle purchased in the Netherlands. '
        || 'Price tier: ' || PRICE_TIER || '. Buyer from: ' || REVIEWER_CITY || '. '
        || 'Usage: ' || USAGE_TYPE || '. Purchase type: ' || PURCHASE_TYPE || '. '
        || CASE 
            WHEN IS_PON_BRAND THEN 'This is a Pon.Bike brand. Common themes: '
                || CASE WHEN BIKE_CATEGORY IN ('E-bike City', 'E-bike Trekking') THEN 'battery range concerns, motor noise, good Dutch design, dealer availability varies by brand.' 
                       WHEN BIKE_CATEGORY = 'Cargo' THEN 'heavy but practical, turning radius, family friendly, quality build.' 
                       WHEN BIKE_CATEGORY IN ('Road', 'Mountain Bike', 'Gravel') THEN 'good components, frame quality, weight, value for money vs Trek/Specialized.' 
                       ELSE 'comfort, design, practical features, value proposition.' END
            ELSE 'This is a competitor brand. '
                || CASE WHEN BRAND_NAME IN ('Trek', 'Specialized') THEN 'Known for innovation, strong dealer network, premium experience, reliable components.'
                       WHEN BRAND_NAME IN ('Giant', 'Cube', 'Merida') THEN 'Great value for money, solid engineering, wide range.'
                       WHEN BRAND_NAME IN ('Batavus', 'Sparta', 'Cortina') THEN 'Dutch heritage, practical commuter bikes, good dealer network, reliable.'
                       WHEN BRAND_NAME IN ('Riese & Muller') THEN 'Premium cargo and e-bikes, excellent build quality, expensive but worth it.'
                       ELSE 'Established brand with loyal following.' END
           END
        || CASE 
            WHEN RATING = 5 THEN ' Write an enthusiastic review praising specific bike features (components, ride quality, battery, design).'
            WHEN RATING = 4 THEN ' Write a positive review with one specific complaint (weight, price, delivery, component).'
            WHEN RATING = 3 THEN ' Write a mixed review mentioning both positives and negatives about the bike.'
            WHEN RATING = 2 THEN ' Write a negative review with specific product complaints (defects, poor components, misleading specs).'
            ELSE ' Write a very negative review detailing multiple product problems and poor experience.'
           END
        || ' Be specific about bicycle features. Do not use generic phrases. Return ONLY the review text.'
    ) AS REVIEW_TEXT,
    PURCHASE_TYPE,
    USAGE_TYPE
FROM REVIEWS_STAGING
""").collect()

print(f"Generated {session.table('REVIEWS').count()} AI-written reviews")

In [ ]:
session.sql("DROP TABLE IF EXISTS REVIEWS_STAGING").collect()
print("Cleaned up staging table.")

## Step 8: Validate the Data

In [ ]:
print("=== Data Validation Summary ===\n")

print("Row Counts:")
for table in ["DIM_BRANDS", "DIM_BIKE_CATEGORIES", "BRIDGE_BRAND_CATEGORIES", "FACT_BRAND_METRICS", "REVIEWS"]:
    count = session.table(table).count()
    print(f"  {table}: {count}")

print("\n--- Rating Distribution ---")
session.sql("""
    SELECT 
        RATING,
        COUNT(*) AS COUNT,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS PERCENTAGE
    FROM REVIEWS
    GROUP BY RATING
    ORDER BY RATING DESC
""").show()

print("\n--- Average Ratings: Pon Brands vs Competitors ---")
session.sql("""
    SELECT 
        CASE WHEN b.IS_PON_BRAND THEN 'Pon Brands' ELSE 'Competitors' END AS OWNERSHIP,
        ROUND(AVG(r.RATING), 2) AS AVG_RATING,
        COUNT(*) AS REVIEW_COUNT
    FROM REVIEWS r
    JOIN DIM_BRANDS b ON r.BRAND_ID = b.BRAND_ID
    GROUP BY OWNERSHIP
    ORDER BY OWNERSHIP
""").show()

print("\n--- Top 5 Brands by Rating (min 5 reviews) ---")
session.sql("""
    SELECT 
        b.BRAND_NAME,
        b.BRAND_TYPE,
        b.IS_PON_BRAND,
        ROUND(AVG(r.RATING), 2) AS AVG_RATING,
        COUNT(*) AS REVIEW_COUNT
    FROM REVIEWS r
    JOIN DIM_BRANDS b ON r.BRAND_ID = b.BRAND_ID
    GROUP BY b.BRAND_ID, b.BRAND_NAME, b.BRAND_TYPE, b.IS_PON_BRAND
    HAVING COUNT(*) >= 5
    ORDER BY AVG_RATING DESC
    LIMIT 5
""").show()

print("\n--- Monthly Metrics Sample (Pon Brands, 2025) ---")
session.sql("""
    SELECT 
        b.BRAND_NAME,
        m.YEAR_MONTH,
        m.NL_UNITS_SOLD,
        m.NL_REVENUE_EUR,
        m.NL_MARKET_SHARE_PCT
    FROM FACT_BRAND_METRICS m
    JOIN DIM_BRANDS b ON m.BRAND_ID = b.BRAND_ID
    WHERE b.IS_PON_BRAND = TRUE AND m.YEAR_MONTH >= '2025-01-01'
    ORDER BY b.BRAND_NAME, m.YEAR_MONTH
    LIMIT 6
""").show()

print("\n--- Sample Reviews ---")
session.sql("""
    SELECT 
        b.BRAND_NAME,
        r.BIKE_CATEGORY,
        r.RATING,
        r.REVIEW_TITLE,
        LEFT(r.REVIEW_TEXT, 100) || '...' AS REVIEW_PREVIEW
    FROM REVIEWS r
    JOIN DIM_BRANDS b ON r.BRAND_ID = b.BRAND_ID
    ORDER BY RANDOM()
    LIMIT 3
""").show()

print("\n=== Validation Complete ===")